In [1]:
import pltkit
import numpy as np
import pandas as pd
import xarray as xr
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import sys, os, glob, re
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cartopy.io.shapereader as shapereader
from matplotlib.ticker import FuncFormatter
import matplotlib.gridspec as gridspec
sys.path.append(os.path.abspath(".."))
from matplotlib.patches import Rectangle

In [2]:
"""
PARAMS
"""
wdir = "X:/user/liprandicn/projects/mt-comparison/"

## Figure 1

In [3]:
### Add population weighted
### Run code only once!!!

# wdir = "X:\\user\\liprandicn\\Projects\\mt-comparison\\"
# era5_dir = "X:/user/liprandicn/data/ERA5/t2m_daily"
# years=range(1980,2024)

# pop = xr.open_dataset(os.path.dirname(wdir) + f'/data/IMAGE/IMAGE_population/SSP2/GPOP.nc')
    
# # Reduce resolution to 15 min to match ERA5 data
# pop = pop.coarsen(latitude=3, longitude=3, boundary='pad').sum(skipna=True)

# # Select years
# pop = pop.sel(time=slice(f'{years[0]}-01-01', f'{years[-1]}-01-01'))

# temperaturas = [20, 25, 30, 35, 40]
# people_days = {tempe: [] for tempe in temperaturas}


# for year in years:
    
#     print(year)
#     # Read file and shift longitude coordinates
#     era5_daily = xr.open_dataset(era5_dir+f"/era5_t2m_mean_day_{year}.nc")

#     # Shift longitudinal coordinates  
#     era5_daily = era5_daily.assign_coords(longitude=((era5_daily.coords["longitude"] + 180) % 360 - 180)).sortby("longitude")

#     # Convert to Celsius 
#     era5_daily -= 273.15

#     era5_daily = era5_daily.interp(
#         latitude=np.clip(pop.latitude, era5_daily.latitude.min().item(), era5_daily.latitude.max().item()), 
#         method="nearest"
#         )

#     era5_daily=era5_daily.interp(longitude=pop.longitude, method="nearest")
#     era5_daily = era5_daily.drop_vars("number").t2m
    
#     pop_year = pop.sel(time=f"{year}-01-01").GPOP.drop_vars("time")
    
#     for tempe in [20,25,30,35,40]:
#         era_masked = (era5_daily > tempe) * pop_year
#         era_masked = era_masked.sum(dim=["latitude", "longitude", "valid_time"])
#         people_days[tempe].append(era_masked.item())
        
# df_anual = pd.DataFrame(people_days, index=years)

# df_anual["GPOP"] = pop.sum(dim=["latitude", "longitude"]).GPOP.values
# df_anual.to_csv("X:\\user\\liprandicn\\projects\\mt-comparison\\figures\\Paper1\\figure1_people_days_heat.csv")

In [3]:
wdir = "X:\\user\\liprandicn\\Projects\\mt-comparison\\models\\"
colors_list = ["#C8553D", "#566E3D", "#222E50", "#FEA82F", "#829191"]
temp_type = "heat"
age_group = "oldest"
cause = "All causes"
rt = "IMAGE"
var = "mortality"

models = {
    "hon_file" : [
        0,
        "honda2014/output/ComparisonHonda/mortality_ComparisonHonda_SSP2_ERA5_1980-2023_counterfactual",
        "Honda et al., 2014",
        "H"
        ],
    "sco_file" : [
        1,
        "scovronick2024/output/ComparisonScovronick/mortality_ComparisonScovronick_SSP2_ERA5_1980-2023_counterfactual",
        "Scovronick et al., 2024",
        "S"
        ],
    "car_file" : [
        2,
        "carleton2022/output/ComparisonCarletonCounter/mortality_ComparisonCarletonCounter_SSP2_ERA5_NoAdap_1980-2023_*",
        "Carleton et al., 2022",
        "C"
        ],       
    "bur_file" : [
        3,
        "burkart2022/output/ComparisonBurkart/mortality_ComparisonBurkart_SSP2_ERA5_1980-2023_counterfactual_*",
        "Burkart et al., 2021",
        "B"
        ]
    }

In [ ]:
rt="ISO3"
var="relative_mortality"
estimates = []
for i, model in enumerate(models):
    main, lower, upper = pltkit.LoadMortalityDraws(wdir, models[model][1], rt, None, temp_type, cause, age_group, var, None)
    estimates.append(main.drop_vars("quantile", errors="ignore"))
    estimates.append(lower.drop_vars("quantile", errors="ignore"))
    estimates.append(upper.drop_vars("quantile", errors="ignore"))
    
for i in [9,10,11]:
    estimates[i]=estimates[i].drop_vars("region_type")
    
estimates = xr.concat(estimates, dim="model", join="outer", coords="minimal")

In [ ]:
stds = (
    estimates
    # .sel(year=slice(2014,2023))
    .std(dim=["year", "model"])
    .drop_vars(["region_type", "t_type", "age_group", "cause"])
    .to_dataframe()
    .reset_index()
)

ir = gpd.read_file(wdir + "carleton2022/data/CarletonSM/ir_shp/impact-region.shp")
ir["geometry"] = ir["geometry"].make_valid()
ir = ir.dissolve(by="ISO", aggfunc=np.nansum)
ir =ir.merge(stds, left_on="ISO", right_on="region", how="left")
ir.loc[ir["region"] == "ATA", "relative_mortality"] = 0
# ir#.plot(column=var, legend=True)

In [ ]:
fig = plt.figure(figsize=(5,6), dpi=300)

pos_ax1 = [0.1, 0.4, 0.85, 0.4]
pos_ax2 = [0.1, 0.05, 0.85, 0.25]
labelsize=6
titlesize=8
lettersize=7
years_vlines = [1998, 2010, 2016, 2019, 2022]

### -------------- Add upper left figure -------------
ax1 = fig.add_axes(pos_ax1)

temp_type = "heat"
age_group = "oldest"
cause = "All causes"
rt = "IMAGE"
var = "mortality"

for i, model in enumerate(models):
    rt="IMAGE"
    main, lower, upper = pltkit.LoadMortalityDraws(wdir, models[model][1], rt, "World", temp_type, cause, age_group, var, None)
    
    line, = ax1.plot(main.year, main.values, label=models[model][2], linewidth=2, c=colors_list[i], clip_on=False)
    
    poly = ax1.fill_between(main.year, lower.values, upper.values, color=colors_list[i], alpha=0.3)
    poly.set_clip_on(False)

for year in years_vlines:
    ax1.axvline(x=year, color='gray', linestyle='--', linewidth=1, alpha=0.7)
ax1.set_title(f"Global heat mortality trends in the over-65 population", fontsize=titlesize, y=1.03)
ax1.legend(frameon=False, fontsize=6)
ax1.set_ylabel("Global excess mortality", fontsize=labelsize)
ax1.set_ylim(-0.9e5, 6.5e5)
ax1.tick_params(axis='both', labelsize=labelsize)


ax1.yaxis.set_major_formatter(FuncFormatter(lambda x, pos: f'{x/1000:g}k'))
ax1.spines["top"].set_visible(False)
ax1.spines["right"].set_visible(False)

ax1.text(-0.02, 1.07, 'A)', transform=ax1.transAxes, fontsize=lettersize, weight='bold', va='bottom', ha='right')


### ------------- Add lower figures (Separated vertically) -------------
df_anual = pd.read_csv("X:\\user\\liprandicn\\projects\\mt-comparison\\figures\\Paper1\\figure1_people_days_heat.csv", index_col=0)

temperatures = [25, 30, 35]
colors_reds = ["#FFC2C2", "#FF6B6B", "#C90000"]

# Parameters
x_start = pos_ax2[0]
y_start = pos_ax2[1]
width = pos_ax2[2]
total_height = pos_ax2[3]

num_plots = len(temperatures)
gap = 0.015 # Vetrical gap between subplots (in figure coordinates)
sub_height = (total_height - (gap * (num_plots - 1))) / num_plots

for i, t in enumerate(temperatures):
    # Get y position for the current subplot
    y_pos = y_start + (num_plots - 1 - i) * (sub_height + gap)
    
    # Create new axis
    ax_sub = fig.add_axes([x_start, y_pos, width, sub_height])
    
    # Plot data
    y_data = df_anual[f"{t}"] / df_anual["GPOP"]
    ax_sub.plot(df_anual.index, y_data, label=f"{t}°C", c=colors_reds[i], linewidth=2)
    
    for year in years_vlines:
        ax_sub.axvline(x=year, color='gray', linestyle='--', linewidth=1, alpha=0.7)
    
    # Configure axis
    ax_sub.spines["top"].set_visible(False)
    ax_sub.spines["right"].set_visible(False)
    ax_sub.tick_params(axis='both', labelsize=labelsize)
    ax_sub.set_ylabel(f"{t}°C", fontsize=6)

    
    # Add title only to the first subplot
    if i == 0:
        ax_sub.set_title("Annual individual exposure to heat over a temperature threshold", fontsize=titlesize, y=1.05)
        ax_sub.text(-0.02, 1.3, 'B)', transform=ax_sub.transAxes, fontsize=lettersize, weight='bold', va='bottom', ha='right')
        
    # Remove x-tick labels for all but the last subplot
    if i < num_plots - 1:
        ax_sub.set_xticklabels([])
        
# plt.savefig(wdir+"figures\\Paper1\\figure1.png", dpi=300, bbox_inches='tight')
# plt.savefig(wdir+"figures\\Paper1\\figure1.pdf", bbox_inches='tight')

In [ ]:
fig = plt.figure(figsize=(15,10), dpi=300)

pos_ax1 = [0.1, 0.3, 0.3, 0.25]
pos_ax2 = [0.1, 0.05, 0.3, 0.15]
pos_map = [0.42, 0.0, 0.55, 0.65]
labelsize=7
titlesize=10
lettersize=8


### -------------- Add upper left figure -------------
ax1 = fig.add_axes(pos_ax1)

temp_type = "heat"
age_group = "oldest"
cause = "All causes"
rt = "IMAGE"
var = "mortality"

for i, model in enumerate(models):
    rt="IMAGE"
    main, lower, upper = pltkit.LoadMortalityDraws(wdir, models[model][1], rt, "World", temp_type, cause, age_group, var, None)
    
    line, = ax1.plot(main.year, main.values, label=models[model][2], linewidth=2, c=colors_list[i], clip_on=False)
    
    poly = ax1.fill_between(main.year, lower.values, upper.values, color=colors_list[i], alpha=0.3)
    poly.set_clip_on(False)

for year in years_vlines:
    ax1.axvline(x=year, color='gray', linestyle='--', linewidth=1, alpha=0.7)
ax1.set_title(f"Global heat mortality trends in the over-65 population", fontsize=titlesize, y=1.1)
ax1.legend(frameon=False, fontsize=6)
ax1.set_ylabel("Global excess mortality", fontsize=labelsize)
ax1.set_ylim(-0.9e5, 6.5e5)
ax1.tick_params(axis='both', labelsize=labelsize)


ax1.yaxis.set_major_formatter(FuncFormatter(lambda x, pos: f'{x/1000:g}k'))
ax1.spines["top"].set_visible(False)
ax1.spines["right"].set_visible(False)

ax1.text(-0.02, 1.15, 'A)', transform=ax1.transAxes, fontsize=lettersize, weight='bold', va='bottom', ha='right')


### ------------- Add lower left figure -------------

# ax2 = fig.add_axes(pos_ax2)

# df_anual = pd.read_csv(wdir+"figures\\Paper1\\figure1_people_days_heat.csv", index_col=0)
df_anual = pd.read_csv("X:\\user\\liprandicn\\projects\\mt-comparison\\figures\\Paper1\\figure1_people_days_heat.csv", index_col=0)

temperatures = [25, 30, 35]
colors_reds = ["#FFC2C2", "#FF6B6B", "#C90000"]

# Parameters
x_start = pos_ax2[0]
y_start = pos_ax2[1]
width = pos_ax2[2]
total_height = pos_ax2[3]

num_plots = len(temperatures)
gap = 0.015 # Vetrical gap between subplots (in figure coordinates)
sub_height = (total_height - (gap * (num_plots - 1))) / num_plots

for i, t in enumerate(temperatures):
    # Get y position for the current subplot
    y_pos = y_start + (num_plots - 1 - i) * (sub_height + gap)
    
    # Create new axis
    ax2 = fig.add_axes([x_start, y_pos, width, sub_height])
    
    # Plot data
    y_data = df_anual[f"{t}"] / df_anual["GPOP"]
    ax2.plot(df_anual.index, y_data, label=f"{t}°C", c=colors_reds[i], linewidth=2)
    
    for year in years_vlines:
        ax2.axvline(x=year, color='gray', linestyle='--', linewidth=1, alpha=0.7)
    
    # Configure axis
    ax2.spines["top"].set_visible(False)
    ax2.spines["right"].set_visible(False)
    ax2.tick_params(axis='both', labelsize=labelsize)
    ax2.set_ylabel(f"Days above \n {t}°C", fontsize=6)

    
    # Add title only to the first subplot
    if i == 0:
        ax2.set_title("Annual individual exposure to heat \n over a temperature threshold", 
                      fontsize=titlesize, y=1.5)
        ax2.text(-0.02, 1.8, 'B)', transform=ax2.transAxes, fontsize=lettersize, weight='bold', va='bottom', ha='right')
        
    # Remove x-tick labels for all but the last subplot
    if i < num_plots - 1:
        ax2.set_xticklabels([])

        
### --------------- Add map -------------
ax = fig.add_axes(pos_map, projection=ccrs.Robinson(central_longitude=0), frameon=True) 
ax.spines['geo'].set_linewidth(0.2)
ax.coastlines(resolution='10m', lw = 0.1)

im = ir.plot(
    ax=ax, 
    column="relative_mortality",
    transform=ccrs.PlateCarree(),
    vmax=100,
    zorder=1,
    )
im = ax.collections[-1] 
ax.add_feature(cfeature.OCEAN, facecolor='white', zorder=2)

# Manually set colorbar limits
cbar = fig.colorbar(im, ax=ax, orientation='horizontal', shrink=0.4, pad=0.05, aspect=20)
cbar.set_label('Relative mortality \n [deaths/100,000 people]', fontsize=7)
cbar.ax.tick_params(labelsize=7, length=2, width=0.5)

# Add Antarctica 
land_shp = shapereader.natural_earth(resolution='110m', category='physical', name='land')
land_geoms = shapereader.Reader(land_shp).geometries()
for land in land_geoms:
    
    min_x, min_y, max_x, max_y = land.bounds
    
    ATA = min_y < -60
    GRL = (min_x > -80 and max_x < -5) and (min_y > 55 and max_y < 85)

    if ATA or GRL:
        ax.add_geometries([land], ccrs.PlateCarree(), 
                          facecolor='whitesmoke', 
                          edgecolor='none', 
                          zorder=2)

ax.set_title("Model differencences in heat-related mortality estimates \n Standard deviation of 2014-2023 model estimates", 
             fontsize=titlesize, y=1.05)
ax.text(0.1, 1.1, 'C)', transform=ax.transAxes, fontsize=lettersize, weight='bold', va='bottom', ha='right')
        
        
# plt.savefig(wdir+"figures\\Paper1\\figure1.png", dpi=300, bbox_inches='tight')
# plt.savefig(wdir+"figures\\Paper1\\figure1.pdf", bbox_inches='tight')
plt.show()

## Figure 3

In [ ]:
region_type="IMAGE"
t_type = "heat"
variable="mortality"
age_group = "All ages"
region = "World"
years = range(2000,2101)
cause=None

scenarios = { # "SSP1_L", "SSP1_M", "SSP1_ML",
  "SSP1_VLHO":"SSP1_LN", "SSP1_VLLO":"SSP1_VL",
  "SSP2_L":'SSP2_L', "SSP2_ML":"SSP2_ML", "SSP2_M":"SSP2_M", #"SSP2_VLHO", "SSP2_VLLO",
  "SSP5_HL":"SSP5_HL",
  "SSP3_H":"SSP3_H", #"SSP5_H", 
}

colours = ["#4A7C80", "#B89648", "#B56545", "#7E5E8F", "#557A42", "#5B6E91", "#AD526B", "#6E6E6E"]

fig, ax = plt.subplots(1,2, figsize=(16,8), dpi=300)
ax = ax.flatten()

for i,adap in enumerate(["_ssp", "_NoAdap"]):
  for j,scenario in enumerate(list(scenarios.keys())):
      filename = f"models/Carleton2022/output/ScenarioMIP7/mortality_ScenarioMIP7_{scenario}{adap}*"    
      mean, lower, upper = pltkit.LoadMortalityDraws(wdir, filename, region_type, region, t_type, cause, age_group, variable, years=None, range=[0.25,0.75])
      ax[i].plot(mean.year, mean.values, linewidth=2, alpha=1, c=colours[j], label=scenarios[scenario], clip_on=False)
      fill = ax[i].fill_between(mean.year, lower.values, upper.values, color=colours[j], alpha=0.1)
      fill.set_clip_on(False)
      
      ax[0].set_ylim(-0.5e5, 1.5e6)
      ax[1].set_ylim(-0.5e6, 13e6)
      
      ax[0].set_ylabel("Global heat-related mortality (million people)", fontsize=13)
      ax[i].yaxis.set_major_formatter(FuncFormatter(lambda x, pos: f'{x/1e6:g}M'))
      ax[i].spines["top"].set_visible(False)
      ax[i].spines["right"].set_visible(False)
      
      ax[0].set_title("Full adaptation", fontsize=14)
      ax[1].set_title("No adaptation", fontsize=14)


leg = plt.legend(frameon=False, fontsize=12, bbox_to_anchor=(0.25,1))
for handle, text, color in zip(leg.legend_handles, leg.get_texts(), colours):
    text.set_color(color)
    # text.set_weight('bold')
    handle.set_visible(False)
    
# plt.suptitle("Heat-related mortality projections under ScenarioMIP7", y=1.01)
plt.show()

In [ ]:
region_type="IMAGE"
t_type = "heat"
variable="mortality"
age_group = "All ages"
region = "World"
years = range(2000,2101)
cause=None

scenarios = [
  "ssp245_r1", "ssp245_r2", "ssp245_r3", "ssp370_r1", "ssp370_r2", "ssp370_r3", "ssp585_r1", "ssp585_r2", "ssp585_r3"
]

colours = ["C0", "C1", "C2", "C3", "C4", "C5", "C6", "C7", "C8", "C9", "C10", "C11", "C12", "C13"]

for i,scenario in enumerate(scenarios):
    file_list = sorted(glob.glob(wdir+f"models/Carleton2022/output/ScenarioMIP7/mortality_ScenarioMIP7_SSP3_H_{scenario}*.nc"))
    for j in range(len(file_list)):
      filename = re.search(r'([^\\/]+)\.nc$', file_list[j]).group(1) 
      ds = pltkit.LoadMortality(wdir, filename, region_type, region, t_type, cause, age_group, variable)
      plt.plot(ds.year, ds.values, linewidth=0.4, c=colours[i], alpha=0.5)

plt.legend()
# plt.title("LHS 50 draws - SSP2_M_CP")
plt.show()

In [ ]:
region_type="IMAGE"
t_type = "heat"
variable="mortality"
age_group = "All ages"
region = "World"
years = range(2000,2101)
cause=None

scenarios = [
  "ssp245_r1", "ssp245_r2", "ssp245_r3", "ssp370_r1", "ssp370_r2", "ssp370_r3", "ssp585_r1", "ssp585_r2", "ssp585_r3"
]

colours = ["C0", "C1", "C2", "C3", "C4", "C5", "C6", "C7", "C8", "C9", "C10", "C11", "C12", "C13"]

for i,scenario in enumerate(scenarios):
    file_list = sorted(glob.glob(wdir+f"models/Carleton2022/output/ScenarioMIP7/mortality_ScenarioMIP7_SSP3_H_NoAdap_{scenario}_*.nc"))
    for j in range(len(file_list)):
      filename = re.search(r'([^\\/]+)\.nc$', file_list[j]).group(1) 
      ds = pltkit.LoadMortality(wdir, filename, region_type, region, t_type, cause, age_group, variable)
      plt.plot(ds.year, ds.values, linewidth=0.4, c=colours[i], alpha=0.5)

plt.legend()
# plt.title("LHS 50 draws - SSP2_M_CP")
plt.show()

## Figure 4

In [ ]:
regions = {"CHN":"China region", "INDIA":"India", "WEU":"Western Europe", 
           "USA":"United States", "BRA":'Brazil', "SAF":"South Africa"}
predictors = ["ssp", "climate", "variability", "erf"]

dfs = {}
for region in list(pltkit.IMAGE_REGIONS.keys()) + ["World"]:
    dfs[region] = pd.read_parquet(
        wdir
        + f"models/carleton2022/output/ScenarioMIP7/Shapley/ShapleyPlot_{region}_All ages.parquet"
    )

colours = {
    "Plot_Avg_SD": "#FF000000",  # Transparente
    "Plot_Avg_SD_Surrogate": "#9cd993",
    "Plot_ssp": "#9cd993",
    "Plot_climate": "#fa9e50",
    "Plot_variability": "#226fac",
    "Plot_erf": "#c9263d",
    "Plot_Resid": "#7a5cf8",
}

fig = plt.figure(figsize=(10, 13), dpi=300)
gs = gridspec.GridSpec(4, 2, figure=fig)

ax0 = fig.add_subplot(gs[0, :])
ax0.set_title("World")

ax0 = (
    dfs["World"][
        ["Year", "Plot_Avg_SD", "Plot_Avg_SD_Surrogate"]
        + [f"Plot_{p}" for p in predictors]
        + ["Plot_Resid"]
    ]
    .set_index("Year")
    .plot(
        kind="bar",
        stacked=True,
        color=colours,
        edgecolor="none",
        width=0.45,
        ax=ax0,
        legend=False,
    )
)

dfs["World"][["Average"]].plot(
    kind="line", color="k", ax=ax0, legend=False,
)

dfs["World"][["Median"]].plot(
    kind="line", linestyle="--", color="k", ax=ax0, legend=False
)
ax0.set_xticks(ax0.get_xticks()[::10])
ax0.tick_params(axis="x", rotation=0)
ax0.set_xlabel("")
ax0.spines[["top", "right"]].set_visible(False)
ax0.set_ylabel("Total mortality \n [thousand people]", fontsize=11)
ax0.yaxis.set_major_formatter(FuncFormatter(lambda x, pos: f'{x/1000:,.0f}k'))

handles_raw, labels_raw = ax0.get_legend_handles_labels()
map_legend = {
    "Average": "Average",
    "Plot_ssp": "SSP",
    "Plot_climate": "Climate Change",
    "Median": "Median",
    "Plot_variability": "Climate Variability",
    "Plot_erf": "ERF uncertainty",
}
handles_filtered = []
labels_filtered = []

for h, l in zip(handles_raw, labels_raw):
    if l in map_legend:
        handles_filtered.append(h)
        labels_filtered.append(map_legend[l])

ax0.legend(
    handles_filtered,
    labels_filtered,
    loc="upper left",
    fontsize=8,
    frameon=False,
    ncol=2,
)


axes = []
for i in range(1, 4):
    for j in range(2):
        if len(axes) < 6:
            ax = fig.add_subplot(gs[i, j])
            axes.append(ax)

for i, ax in enumerate(axes):
    region = list(regions.keys())[i]
    df = dfs[region]

    df[
        ["Year", "Plot_Avg_SD", "Plot_Avg_SD_Surrogate"]
        + [f"Plot_{p}" for p in predictors]
        + ["Plot_Resid"]
    ].set_index("Year").plot(
        kind="bar",
        stacked=True,
        color=colours,
        edgecolor="none",
        width=0.6,
        title=f"{regions[region]}",
        ax=ax,
        legend=False,
    )

    df[["Average"]].plot(
        kind="line", color="k", ax=ax, legend=False, linewidth=1
    )

    df[["Median"]].plot(
        kind="line", linestyle="--", color="k", ax=ax, legend=False, linewidth=1
    )

    ax.set_xticks(ax.get_xticks()[::20])
    ax.set_xlabel("")
    # Remove top and right spines for all regional panels
    ax.spines[["top", "right"]].set_visible(False)
    
    if i in [0, 2, 4]:
        ax.set_ylabel("Total mortality \n [thousand people]", fontsize=10)
    
    ax.yaxis.set_major_formatter(FuncFormatter(lambda x, pos: f'{x/1000:,.1f}k'))


fig.tight_layout()
plt.show()